In [ ]:
# ============================================================
# CONFIGURACIÓN DE ENTORNO
# ============================================================
# Cambia LOCAL_DATA_DIR si vas a ejecutar esto como Jupyter LOCAL
# (no Colab), leyendo los ficheros directamente desde tu disco --
# esto es lo más rápido posible, sin ninguna subida/copia de por
# medio. Déjalo en None para usar Google Drive en Colab en su lugar.
LOCAL_DATA_DIR = r"C:\Proyectos\Datasets\Dataset_05082026"  # o None

import os

if LOCAL_DATA_DIR and os.path.isdir(LOCAL_DATA_DIR):
    DATA_DIR = LOCAL_DATA_DIR
    print(f"[INFO] Modo LOCAL: leyendo directamente desde {DATA_DIR}")
else:
    # Entorno Colab: se monta Google Drive (mucho más rápido que
    # files.upload() para ficheros de cientos de MB -- súbelos UNA
    # vez a tu Drive y luego se leen directamente desde ahí, sin
    # transferirlos otra vez por el navegador cada vez que abras el
    # notebook).
    from google.colab import drive
    drive.mount('/gdrive')
    DATA_DIR = "/gdrive/MyDrive/Datasets"  # ajusta a tu carpeta real en Drive
    print(f"[INFO] Modo Colab + Drive: leyendo desde {DATA_DIR}")
    print("[AVISO] Si los CSV no están ahí, cópialos una vez a esa carpeta de Drive")
    print("        (arrastrar en drive.google.com es mucho más rápido que")
    print("        files.upload(), que sube por el navegador y por eso iba tan lento).")

In [ ]:
# ============================================================
# ESCENARIOS DE AGREGACIÓN — a partir de datos REALES con IP
# ============================================================
# Objetivo: cerrar el hueco que dejó pendiente model_v1 (ver README de
# MODEXRE / Capítulo 3-4 de la memoria): entrenar de verdad las
# variables agg_distinct_dst_ports / agg_distinct_dst_hosts /
# agg_events_in_window, en vez de dejarlas a 0 constante en
# entrenamiento (que es lo que ocurría al no usar nunca
# enrich_with_aggregation() sobre los datos de laboratorio).
#
# IMPORTANTE — esto NO fabrica escenarios sintéticos ficticios: agrupa
# las filas REALES (ya con srcip/source_ip real, gracias a la
# corrección de los notebooks de limpieza) por IP de origen, las
# ordena por tiempo real, y calcula la señal de agregación tal como lo
# haría el pipeline formal sobre evidencia real. Así, un ataque de
# tipo Reconnaissance (barrido de muchos hosts, pocos puertos) y uno
# de tipo PortScan (un host, muchos puertos) generan, de forma
# natural, agg_distinct_dst_hosts y agg_distinct_dst_ports distintos
# -- sin necesidad de inventar nada.
#
# Salida por dataset: <nombre>_scenarios_features.csv con el MISMO
# esquema de columnas que espera model_v1 (ver
# models_certified/model_v1.manifest.json), más agg_events_per_second
# como columna adicional (aditiva, ver flow_aggregation.py del
# proyecto para el porqué).

!pip install -q pandas numpy tqdm

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

RANDOM_STATE = 42
WINDOW_SECONDS = 60.0
MIN_OBSERVED_SECONDS = 1.0  # mismo suelo que backend/app/features/flow_aggregation.py

# ------------------------------------------------------------
# Configuración por dataset: qué columnas leer y cómo renombrarlas
# al esquema común que usa model_v1.
# ------------------------------------------------------------
DATASET_CONFIGS = {
    "unsw": {
        "path": os.path.join(DATA_DIR, "UNSW_NB15_full_clean_v2.csv"),
        "usecols": ["attack_cat", "label", "srcip", "dstip", "sport", "dsport",
                    "stime", "dur", "spkts", "dpkts"],
        "src_ip_col": "srcip",
        "dst_ip_col": "dstip",
        "dst_port_col": "dsport",
        "time_col": "stime",
        "time_is_epoch": True,          # stime en UNSW-NB15 ya es epoch (segundos)
        "duration_col": "dur",
        "packets_out_col": "spkts",     # origen -> destino
        "packets_in_col": "dpkts",      # destino -> origen
        "out_path": "/content/UNSW_NB15_scenarios_features.csv" if not (LOCAL_DATA_DIR and os.path.isdir(LOCAL_DATA_DIR)) else os.path.join(DATA_DIR, "UNSW_NB15_scenarios_features.csv"),
    },
    "cicids": {
        "path": os.path.join(DATA_DIR, "MachineLearningCVE_full_clean_v2.csv"),
        "usecols": ["attack_cat", "label", "source_ip", "destination_ip", "destination_port",
                    "timestamp", "flow_duration", "total_fwd_packets", "total_backward_packets"],
        "src_ip_col": "source_ip",
        "dst_ip_col": "destination_ip",
        "dst_port_col": "destination_port",
        "time_col": "timestamp",
        "time_is_epoch": False,         # timestamp en CICIDS es texto, hay que parsearlo
        "duration_col": "flow_duration",  # en microsegundos (CICFlowMeter)
        "packets_out_col": "total_fwd_packets",
        "packets_in_col": "total_backward_packets",
        "out_path": "/content/MachineLearningCVE_scenarios_features.csv" if not (LOCAL_DATA_DIR and os.path.isdir(LOCAL_DATA_DIR)) else os.path.join(DATA_DIR, "MachineLearningCVE_scenarios_features.csv"),
    },
}


def build_scenario_features(cfg: dict, dataset_label: str) -> pd.DataFrame:
    print(f"\n{'='*60}\n{dataset_label}\n{'='*60}")

    print(">> Leyendo columnas necesarias...")
    df = pd.read_csv(cfg["path"], usecols=cfg["usecols"], low_memory=False)
    print(f"[INFO] Filas leídas: {len(df):,}")

    # --- normalizar tipos ---
    df = df.rename(columns={
        cfg["src_ip_col"]: "_src_ip",
        cfg["dst_ip_col"]: "_dst_ip",
        cfg["dst_port_col"]: "_dst_port",
        cfg["time_col"]: "_time_raw",
        cfg["duration_col"]: "duration",
        cfg["packets_out_col"]: "packets_out",
        cfg["packets_in_col"]: "packets_in",
    })

    if cfg["time_is_epoch"]:
        df["_ts"] = pd.to_numeric(df["_time_raw"], errors="coerce")
    else:
        # CICFlowMeter exporta el timestamp como texto; el formato varía
        # según la versión/exportación (día/mes/año, con o sin segundos).
        # dayfirst=True porque el dataset se generó en formato europeo.
        parsed = pd.to_datetime(df["_time_raw"], errors="coerce", dayfirst=True)
        df["_ts"] = parsed.astype("int64") // 10**9
        df.loc[parsed.isna(), "_ts"] = np.nan

    n_before = len(df)
    df = df.dropna(subset=["_src_ip", "_ts"]).copy()
    n_after = len(df)
    print(f"[INFO] Filas descartadas por falta de IP/timestamp interpretable: {n_before - n_after:,} "
          f"({(n_before - n_after) / max(n_before,1) * 100:.2f}%)")

    df["_src_ip"] = df["_src_ip"].astype(str)
    df["_dst_ip"] = df["_dst_ip"].astype(str)
    df["duration"] = pd.to_numeric(df["duration"], errors="coerce").fillna(0.0)
    df["packets_out"] = pd.to_numeric(df["packets_out"], errors="coerce").fillna(0.0)
    df["packets_in"] = pd.to_numeric(df["packets_in"], errors="coerce").fillna(0.0)

    # --- orden por origen y tiempo: imprescindible para la ventana causal ---
    print(">> Ordenando por origen y tiempo...")
    df = df.sort_values(["_src_ip", "_ts"], kind="mergesort").reset_index(drop=True)

    # --- agregación causal eficiente (contadores incrementales, O(n) real) ---
    # NOTA: esto calcula lo mismo que enrich_with_aggregation() del
    # backend (misma semántica: solo eventos pasados o simultáneos, sin
    # mirar al futuro). La diferencia con la versión del backend
    # (pensada para lotes pequeños de evidencia real, donde una
    # implementación simple es suficiente) es que aquí se procesan
    # millones de filas de datasets de entrenamiento: en vez de
    # recalcular el conjunto de puertos/hosts distintos desde cero en
    # cada fila (O(tamaño de ventana) por fila, inasumible a este
    # volumen -- la primera versión de este script tardaba varias
    # horas), se mantienen contadores incrementales (Counter) que se
    # actualizan en O(1) amortizado al añadir el evento actual y
    # expulsar los que caducan de la ventana.
    from collections import defaultdict

    n = len(df)
    agg_ports = np.zeros(n, dtype=np.int32)
    agg_hosts = np.zeros(n, dtype=np.int32)
    agg_count = np.zeros(n, dtype=np.int32)
    agg_rate = np.zeros(n, dtype=np.float64)

    src_arr = df["_src_ip"].to_numpy()
    ts_arr = df["_ts"].to_numpy()
    dport_arr = df["_dst_port"].to_numpy()
    dhost_arr = df["_dst_ip"].to_numpy()

    group_start = 0
    port_counts = defaultdict(int)
    host_counts = defaultdict(int)

    for i in tqdm(range(n), desc=f"Agregación causal ({dataset_label})", unit="filas", mininterval=1.0):
        if i > 0 and src_arr[i] != src_arr[i - 1]:
            # nuevo origen: reiniciar contadores y ventana
            port_counts.clear()
            host_counts.clear()
            group_start = i

        port_counts[dport_arr[i]] += 1
        host_counts[dhost_arr[i]] += 1

        left = group_start
        while ts_arr[left] < ts_arr[i] - WINDOW_SECONDS:
            p = dport_arr[left]
            port_counts[p] -= 1
            if port_counts[p] == 0:
                del port_counts[p]
            h = dhost_arr[left]
            host_counts[h] -= 1
            if host_counts[h] == 0:
                del host_counts[h]
            left += 1
        group_start = left  # el puntero nunca retrocede dentro del mismo origen

        count = i - left + 1
        agg_ports[i] = len(port_counts)
        agg_hosts[i] = len(host_counts)
        agg_count[i] = count

        observed_span = max(ts_arr[i] - ts_arr[left], 0.0)
        observed_span = max(observed_span, MIN_OBSERVED_SECONDS)
        agg_rate[i] = count / observed_span

    df["agg_distinct_dst_ports"] = agg_ports
    df["agg_distinct_dst_hosts"] = agg_hosts
    df["agg_events_in_window"] = agg_count
    df["agg_events_per_second"] = np.round(agg_rate, 6)

    out = df[["attack_cat", "label", "duration", "packets_in", "packets_out",
              "agg_distinct_dst_hosts", "agg_distinct_dst_ports",
              "agg_events_in_window", "agg_events_per_second"]].copy()

    out.to_csv(cfg["out_path"], index=False, encoding="utf-8")
    print(f"[OK] Guardado → {cfg['out_path']}")
    print(f"     Shape: {out.shape}")
    print(f"     Distribución de attack_cat:")
    print(out["attack_cat"].value_counts())

    return out

In [ ]:
df_unsw = build_scenario_features(DATASET_CONFIGS["unsw"], "UNSW-NB15")

In [ ]:
df_cicids = build_scenario_features(DATASET_CONFIGS["cicids"], "CICIDS2017 (MachineLearningCVE)")

In [ ]:
# ============================================================
# COMBINAR ambos datasets en un único fichero de entrenamiento
# ============================================================
# Mismo esquema de columnas en ambos (attack_cat, label, duration,
# packets_in, packets_out, agg_distinct_dst_hosts,
# agg_distinct_dst_ports, agg_events_in_window,
# agg_events_per_second), así que se pueden concatenar directamente.
# Este es el fichero que hay que subir a la pestaña Laboratorio de
# MODEXRE para certificar el nuevo modelo (model_v2).

COMBINED_PATH = os.path.join(DATA_DIR, "combined_scenarios_features.csv") if (LOCAL_DATA_DIR and os.path.isdir(LOCAL_DATA_DIR)) else "/content/combined_scenarios_features.csv"
MAX_PER_CLASS = 40_000  # tope por clase para no desbalancear ni disparar el tamaño

df_all = pd.concat([df_unsw, df_cicids], ignore_index=True)

parts = []
for cat, grp in df_all.groupby("attack_cat"):
    if len(grp) > MAX_PER_CLASS:
        grp = grp.sample(n=MAX_PER_CLASS, random_state=RANDOM_STATE)
    parts.append(grp)
df_combined = pd.concat(parts, ignore_index=True).sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)

df_combined.to_csv(COMBINED_PATH, index=False, encoding="utf-8")
print(f"[OK] Combinado guardado → {COMBINED_PATH}")
print(f"     Shape: {df_combined.shape}")
print(f"     Peso aproximado: {__import__('os').path.getsize(COMBINED_PATH) / (1024*1024):.1f} MB")
print(f"     Distribución final de attack_cat:")
print(df_combined["attack_cat"].value_counts())

In [ ]:
# ============================================================
# VERSIÓN COMPATIBLE CON LA TAXONOMÍA CERRADA DE MODEXRE
# ============================================================
# app/ocsf/mappers.py valida attack_cat contra una lista CERRADA de
# 13 categorías (ALLOWED_ATTACK_CATEGORIES). Las clases "extra" que
# recuperamos en la limpieza (antes se perdían dentro de "Generic":
# Bot, Web Attack – Xss, Web Attack – Sql Injection, Heartbleed,
# Infiltration, Worms) NO pertenecen a esa lista, y MODEXRE las
# rechaza con TaxonomyError al ingerir el CSV en la pestaña
# Laboratorio.
#
# Esto NO deshace la corrección anterior: combined_scenarios_features
# .csv (el fichero completo, con todas las clases) se conserva tal
# cual para archivo/reproducibilidad -- documenta fielmente qué
# categorías existen realmente en los datos de origen. Este segundo
# fichero es una PROYECCIÓN de ese mismo dataset sobre las 13 clases
# que MODEXRE sabe entrenar y clasificar hoy, colapsando las que
# sobran de vuelta a "Generic" (que es precisamente para lo que
# existe esa categoría: lo que no encaja en ninguna más específica).

MODEXRE_ALLOWED = {
    "Analysis", "Backdoors", "BruteForce", "DDoS", "DoS", "Exploits",
    "Fuzzers", "Generic", "MitM", "Normal", "PortScan", "Reconnaissance",
    "Shellcode",
}

df_modexre = df_combined.copy()
fuera_taxonomia = sorted(set(df_modexre["attack_cat"]) - MODEXRE_ALLOWED)
if fuera_taxonomia:
    print(f"[INFO] Clases fuera de la taxonomía de MODEXRE (se colapsan a 'Generic'): {fuera_taxonomia}")
    df_modexre["attack_cat"] = df_modexre["attack_cat"].apply(
        lambda x: x if x in MODEXRE_ALLOWED else "Generic"
    )

COMBINED_MODEXRE_PATH = COMBINED_PATH.replace(".csv", "_modexre13.csv")
df_modexre.to_csv(COMBINED_MODEXRE_PATH, index=False, encoding="utf-8")
print(f"[OK] Versión compatible con MODEXRE guardada → {COMBINED_MODEXRE_PATH}")
print(f"     >>> ESTE es el fichero que hay que subir a la pestaña Laboratorio <<<")
print(f"     Shape: {df_modexre.shape}")
print(f"     Distribución final de attack_cat (ya dentro de las 13 clases permitidas):")
print(df_modexre["attack_cat"].value_counts())

In [ ]:
# Comprobación específica: ¿Reconnaissance y PortScan ya se distinguen
# de verdad por su patrón de agregación, o siguen pareciéndose?
# (se comprueba sobre df_modexre, que es el que se va a entrenar)
check_cols = ["agg_distinct_dst_hosts", "agg_distinct_dst_ports", "agg_events_in_window"]
print(df_modexre[df_modexre["attack_cat"].isin(["Reconnaissance", "PortScan"])]
      .groupby("attack_cat")[check_cols].median())
print("\n(Se espera: Reconnaissance con agg_distinct_dst_hosts más alto que PortScan,")
print(" y PortScan con agg_distinct_dst_ports más alto que Reconnaissance)")

In [ ]:
if not (LOCAL_DATA_DIR and os.path.isdir(LOCAL_DATA_DIR)):
    from google.colab import files
    files.download(DATASET_CONFIGS["unsw"]["out_path"])
    files.download(DATASET_CONFIGS["cicids"]["out_path"])
    files.download(COMBINED_PATH)
    files.download(COMBINED_MODEXRE_PATH)
else:
    print(f"[INFO] Modo local: los ficheros ya están en disco, no hace falta descargar nada.")
    print(f"       {DATASET_CONFIGS['unsw']['out_path']}")
    print(f"       {DATASET_CONFIGS['cicids']['out_path']}")
    print(f"       {COMBINED_PATH}")
    print(f"       {COMBINED_MODEXRE_PATH}  <-- este es el que subes a Laboratorio")